In [ ]:
import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
from lunanav.constants import R_MOON
from lunanav.visualization import moon_surface, visualize_trajectory

## Test 1: moon_surface with offset [0, 0, 0]
Moon centered at origin, extends from -R_MOON to +R_MOON in each direction

In [ ]:
surf1 = moon_surface(radius=R_MOON, offset=np.array([0.0, 0.0, 0.0]), resolution=25)
print(f"Moon surface created with offset [0, 0, 0]")
print(f"  Surface z range: {surf1.z.min():.0f} to {surf1.z.max():.0f}")
print(f"  Expected z range: ~{-R_MOON:.0f} to ~{R_MOON:.0f}")

## Test 2: moon_surface with offset [0, 0, -R_MOON]
Moon shifted down by R_MOON, extends from -2*R_MOON to 0

In [ ]:
surf2 = moon_surface(radius=R_MOON, offset=np.array([0.0, 0.0, -R_MOON]), resolution=25)
print(f"Moon surface created with offset [0, 0, -R_MOON]")
print(f"  Surface z range: {surf2.z.min():.0f} to {surf2.z.max():.0f}")
print(f"  Expected z range: ~{-2*R_MOON:.0f} to ~0")

## Create simple test trajectory
100-step descent from 10km altitude to 1km altitude at origin

In [ ]:
n_steps = 100
trajectory = np.zeros((n_steps, 13))

# Position: descend from 10km altitude to 1km altitude at origin
for i in range(n_steps):
    t_frac = i / (n_steps - 1)
    # Start at (0, 0, R_MOON + 10000), end at (0, 0, R_MOON + 1000)
    altitude = 10000 * (1 - t_frac) + 1000 * t_frac
    trajectory[i, 0:3] = [0, 0, R_MOON + altitude]
    
    # Velocity: descending
    trajectory[i, 3:6] = [0, 0, -10]  # constant downward velocity
    
    # Quaternion: identity (no rotation)
    trajectory[i, 6:10] = [1, 0, 0, 0]
    
    # Angular velocity: zero
    trajectory[i, 10:13] = [0, 0, 0]

print(f"Created test trajectory with {n_steps} steps")
print(f"  Initial position: {trajectory[0, 0:3]}")
print(f"  Final position: {trajectory[-1, 0:3]}")
print(f"  Shape: {trajectory.shape}")

## Test visualize_trajectory with single trajectory
Moon offset=[0, 0, 0], trajectory centered at origin with 10km descent

In [ ]:
dt = 0.1  # 100ms timestep
t = np.arange(n_steps) * dt

fig = visualize_trajectory(
    trajectory,
    t=t,
    dt=dt,
    title="Test: Simple Descent Trajectory (Default Offset)",
    moon_resolution=30,
    downsample_rate=5
)
fig.show()

## Debug: Trajectory bounds analysis

In [ ]:
offset = np.zeros(3)  # default offset
rel_pos = trajectory[:, 0:3] - offset

print(f"Trajectory bounds (in inertial frame):")
print(f"  X: {trajectory[:, 0].min():.0f} to {trajectory[:, 0].max():.0f}")
print(f"  Y: {trajectory[:, 1].min():.0f} to {trajectory[:, 1].max():.0f}")
print(f"  Z: {trajectory[:, 2].min():.0f} to {trajectory[:, 2].max():.0f}")
print(f"\nTrajectory bounds (relative to moon center):")
print(f"  X: {rel_pos[:, 0].min():.0f} to {rel_pos[:, 0].max():.0f}")
print(f"  Y: {rel_pos[:, 1].min():.0f} to {rel_pos[:, 1].max():.0f}")
print(f"  Z: {rel_pos[:, 2].min():.0f} to {rel_pos[:, 2].max():.0f}")
print(f"\nMoon radius: {R_MOON:.0f}")
print(f"Moon surface z range (with offset [0,0,0]): {-R_MOON:.0f} to {R_MOON:.0f}")
print(f"Trajectory altitude above surface: {trajectory[0, 2] - R_MOON:.0f} to {trajectory[-1, 2] - R_MOON:.0f} m")

## Test with multiple trajectories

In [ ]:
trajectory2 = trajectory.copy()
trajectory2[:, 0] = 1000 * np.sin(np.linspace(0, 2*np.pi, n_steps))  # sinusoidal x motion

fig = visualize_trajectory(
    [trajectory, trajectory2],
    t=t,
    dt=dt,
    title="Test: Multiple Trajectories",
    moon_resolution=30,
    downsample_rate=5
)
fig.show()

## Debug: Trajectory bounds with offset

In [ ]:
offset = np.array([0.0, 0.0, -R_MOON])
rel_pos = trajectory[:, 0:3] - offset

print(f"Trajectory bounds (in inertial frame):")
print(f"  X: {trajectory[:, 0].min():.0f} to {trajectory[:, 0].max():.0f}")
print(f"  Y: {trajectory[:, 1].min():.0f} to {trajectory[:, 1].max():.0f}")
print(f"  Z: {trajectory[:, 2].min():.0f} to {trajectory[:, 2].max():.0f}")
print(f"\nTrajectory bounds (relative to moon center):")
print(f"  X: {rel_pos[:, 0].min():.0f} to {rel_pos[:, 0].max():.0f}")
print(f"  Y: {rel_pos[:, 1].min():.0f} to {rel_pos[:, 1].max():.0f}")
print(f"  Z: {rel_pos[:, 2].min():.0f} to {rel_pos[:, 2].max():.0f}")
print(f"\nMoon radius: {R_MOON:.0f}")
print(f"Moon center at offset: {offset}")